# Week 14: Mini Project — Sensor Log Summary — PHASE 3: Build a checkable report

*📚 Computer Programming I · ⏱️ 5 Hours · 👨‍🏫 Dr. Arif Solmaz*

## Bring the sensor report together

Each week supplied part of this workflow: represent a value, apply a rule, process observations, write reusable calculations, handle failures and save the result.

The project supplies 20 records from temperature, humidity and pressure sensors. Build the existing sensor-log summary using these records and the stated validation rules.

**Try this first — before code.** Choose one valid record and one faulty record from the supplied file. Sketch where each should go: read → validate → group → summarise → save. Decide how you will check the result.

**Why this week's tool?** Use the Python tools you already know to connect the stages. The main work is agreeing on their inputs and outputs and checking that information survives each step.

**By the end.** Walk another person through the cleaned CSV, summary and one handled fault. Explain one limitation of the result using evidence from your own run.


<details><summary>Learning objectives</summary>

## 🎯 Learning Objectives

By the end of this week, you will be able to:

- Apply file I/O and CSV skills in a real-world project
- Read and parse CSV sensor data from a file
- Validate and clean messy data with error handling
- Compute statistics (min, max, mean, count) per sensor
- Generate formatted text reports
- Write cleaned data and summaries to output files
- Combine multiple functions into a complete program

</details>


<details><summary>Class participation and assessment</summary>

---
## 🤝 Mechatronics Learning Contract

- **Professional relevance:** examples and core exercises model the data, sensing, automation, numerical, and decision tasks used in mechatronics engineering.
- **Interaction:** predict before running, compare reasoning with a partner, and ask whenever a step is unclear; scheduled checkpoints guarantee question time.
- **Assessment alignment:** worked examples and Core Exercises 1–10 rehearse the same reasoning operations used on exams—trace, implement, debug, interpret, and justify—while exam values and contexts may change.
- **Learning evidence:** weekly notebooks remain private practice. Non-exam evidence comes from scheduled in-class project demonstrations/presentations using a published rubric, not homework collection.

</details>


<details><summary>Class schedule and checkpoints</summary>

---
## 🧭 Five-Hour Project Roadmap

Build the project during the guided blocks; do not postpone all coding until the end.

| Target | Activity |
|---|---|
| 00:00–00:55 | Read the contract, create the supplied data (EX1), plan the pipeline → Checkpoint 1 |
| 00:55–01:05 | Break |
| 01:05–01:55 | Read records (EX2); inspect malformed and boundary cases → Checkpoint 2 |
| 01:55–02:05 | Break |
| 02:05–02:55 | Validate/clean and calculate sensor statistics (EX3–6) → Checkpoint 3 |
| 02:55–03:05 | Break |
| 03:05–03:55 | Build report and save cleaned CSV (EX7–8) → Checkpoint 4 |
| 03:55–04:05 | Break |
| 04:05–04:45 | Save report, integrate and verify both files (EX9–10) → Checkpoint 5 |
| 04:45–05:00 | Compare results, retry failed cases, and explain the complete pipeline |

EX1–10 are core milestones; EX1 is supplied setup. EX11–12 are optional extensions,
not homework. Worked solutions are available for comparison after your own attempt.
Checkpoints provide private feedback in this runtime, not grades.

</details>


In [1]:
# Run this setup once. These small tools give local study feedback.
_checkpoint_results = {}

def check_answer(number, answer, expected, explanation):
    actual = str(answer).strip().lower().replace(" ", "")
    target = str(expected).strip().lower().replace(" ", "")
    correct = actual == target
    _checkpoint_results[int(number)] = ("Concept check", int(correct), 1)
    if correct:
        print(f"Checkpoint {number}: correct. {explanation}")
    elif not str(answer).strip():
        print(f"Checkpoint {number}: enter your prediction, then run again.")
    else:
        print(f"Checkpoint {number}: review the example and try again.")
    return correct

def record_checkpoint(number, checks):
    """Report each concrete concept check used by the introductory notebook."""
    passed = sum(bool(correct) for _, correct in checks)
    _checkpoint_results[int(number)] = ("Concept checks", passed, len(checks))
    print(f"Checkpoint {number}: {passed}/{len(checks)} concept checks match.")
    for label, correct in checks:
        print(("OK: " if correct else "Review: ") + label)
    return passed, len(checks)

def exercise_checkpoint(number, practiced, expected=8):
    """Summarize an explicit self-report; this does not grade your code."""
    if not isinstance(practiced, (list, tuple, set)):
        _checkpoint_results.pop(int(number), None)
        print("Use a list of exercise numbers, for example [1, 2].")
        return 0, expected
    if any(type(item) is not int or not 1 <= item <= expected for item in practiced):
        _checkpoint_results.pop(int(number), None)
        print(f"Use whole exercise numbers from 1 to {expected}.")
        return 0, expected
    done = set(practiced)
    _checkpoint_results[int(number)] = ("Practice self-report", len(done), expected)
    print(f"Practice self-report: {len(done)}/{expected} core exercises reviewed.")
    print("This is your reflection, not a correctness score or a grade.")
    remaining = [str(i) for i in range(1, expected + 1) if i not in done]
    if remaining:
        print("Still to review:", ", ".join(remaining))
    print("For each exercise: test the result, explain the steps, then compare with the worked solution.")
    return len(done), expected

def show_progress_summary():
    print("\nMy study feedback (this runtime)")
    for number in range(1, 6):
        if number in _checkpoint_results:
            kind, count, total = _checkpoint_results[number]
            print(f"{number}. {kind}: {count}/{total}")
        else:
            print(f"{number}. Not run yet")
    print("These checks send no grading submission. Save your notebook to keep your work.")

print("Local study tools ready.")


Local study tools ready.


## Joining this lesson: records and returned pairs

Recall a small record: `student = {"name": "Elif", "scores": [85, 92]}`.
`student["name"]` reads a key; `student["average"] = 88.5` adds a value;
`"scores" in student` checks whether the key exists. To display the entries, use
`for key, value in student.items(): print(key, value)`.

A function can `return True, "OK"`. Its caller writes `valid, reason = validate(...)`
to unpack the returned tuple. Keep the order consistent: first the decision, then
the explanation. Dictionary keys name fields; tuple positions preserve a short,
agreed order. The complete introduction is in Week 11.

**Türkçe:** Sözlük anahtarı alanın adıdır; demet açma iki sonucu sırayla alır.
Bir kaydı okumadan önce gerekli alanların bulunduğunu kontrol edin.


---
## Part 1: Project Overview

In this project, you will build a **Sensor Log Summary Tool**. Imagine you work at a weather station that collects data from multiple sensors (temperature, humidity, pressure). The data is stored in a CSV file, but some readings are **corrupted or missing**.

Your tool will:
1. **Read** sensor data from a CSV file
2. **Clean** the data by removing bad rows
3. **Compute statistics** for each sensor (min, max, mean, count)
4. **Generate** a formatted summary report
5. **Write** cleaned data to a new CSV file
6. **Write** the summary report to a text file

This project brings together everything you have learned in this course: variables, data types, conditionals, loops, functions, lists, strings, and file I/O.

> 💡 **Note:** Work through each step in order. Each exercise builds on the previous one. By the end, you will combine all steps into one complete program.

---
## Part 2: Project Requirements

### Input
A CSV file called `sensor_data.csv` with three columns:

| Column | Description | Example |
|:---|:---|:---|
| `timestamp` | Date and time of reading | `2024-01-15 08:00` |
| `sensor` | Name of the sensor | `temp`, `humidity`, `pressure` |
| `value` | The sensor reading | `22.5`, `45.2`, `1013.25` |

### Bad Data (to be cleaned)
The file may contain:
- **Empty values** — the value field is blank
- **Non-numeric values** — the value field contains text like `abc`
- **Impossible values** — values that don't make physical sense (e.g., negative humidity)

### Validation Rules

| Sensor | Valid Range |
|:---|:---|
| `temp` | -50.0 to 60.0 °C |
| `humidity` | 0.0 to 100.0 % |
| `pressure` | 800.0 to 1200.0 hPa |

### Output
1. `sensor_data_clean.csv` — cleaned data (bad rows removed)
2. `sensor_report.txt` — formatted summary report with statistics

### Explicit validation and empty-data contract

Every data row must contain exactly three text fields. Strip outer whitespace.
Reject empty fields, an unknown sensor, a malformed timestamp, a non-numeric value,
a nonfinite value (`nan`, `inf`, `-inf`), or a value outside the sensor's inclusive range.
Accept valid negative temperatures. Timestamps use `YYYY-MM-DD HH:MM`; the optional
time filter compares cleaned, validated timestamps inclusively.

An empty file is treated as zero records; a nonempty CSV must have the exact header
`timestamp,sensor,value` (after stripping each header field). A wrong header is a file
error, not a data record. Missing files produce a clear error result in the integrated program.

`validate_row(row)` returns `(True, "OK")` or `(False, reason)`.
`compute_stats(data, sensor_name)` returns `count=0` and `None` for `min`, `max`, `mean`
when a known sensor has no valid readings. Reports display these as `N/A`; never divide by zero.
An unknown sensor requested from `compute_stats` is a `ValueError`.
Both output files must still be written for empty valid input: a header-only CSV and
a report showing zero records and `N/A` statistics.

**Türkçe:** Alan sayısı → boş alan → sensör/zaman → sayıya çevirme → sonluluk → aralık
sırasını izleyin. Hiç veri yoksa ortalama 0 değildir; tanımsızdır (`None`, raporda `N/A`).


### Small library reference for the sensor project

These tools handle file details while your functions express the processing steps.
All belong to Python's standard library; no extra installation is needed.

| Tool | Read it as | Why it appears here |
|---|---|---|
| `csv.reader(file)` | Read one CSV record as a list of text fields | Handles CSV field boundaries, including quoted fields |
| `csv.writer(file).writerows(rows)` | Write a list of records to CSV | Saves the same three-column structure |
| `math.isfinite(value)` | Is this a finite real number? | Rejects `nan`, `inf` and `-inf` after conversion |
| `datetime.strptime(text, format)` | Interpret text using an expected date/time pattern | Rejects invalid dates; `%Y-%m-%d %H:%M` means year-month-day hour:minute |
| `date.strftime(format)` | Turn the date/time back into text | Lets us compare with the required written format |
| `Path(folder) / filename` | Join a folder and filename | Here `/` joins paths; it is not numeric division |
| `path.read_text(encoding="utf-8")` | Read the file's text | Lets a check inspect the saved report |
| `TemporaryDirectory()` | Create a temporary folder for test files | Keeps test fixtures separate from the reference output files |

**Türkçe:** Bu yardımcılar dosya ve biçim ayrıntılarını yönetir. Asıl algoritma yine aynıdır: oku, doğrula, hesapla ve kaydet. `Path` ile kullanılan `/` işareti klasör ile dosya adını birleştirir.


---
## Exercises — Project Milestones

Complete the project stages in order. Exercises 1–10 are the core in-class milestones.


### Core Practice and Optional Extension

- **Exercises 1–10:** core in-class practice.
- **Exercises 11–12:** optional extension; these are not homework.
- At Checkpoint 5, list the exercises you have tested and can explain. This is a self-report, not automatic grading.


---
### ⏱️ Checkpoint 1 of 5 — Pipeline (target 00:55)

Which comes first: clean data or compute statistics?

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [2]:
checkpoint_1_answer = ""  # enter your answer
check_answer(
    1, checkpoint_1_answer, 'clean data',
    'Statistics should use validated, cleaned values.',
)


Checkpoint 1: enter your prediction, then run again.


False

---
## Part 3: Step 1 — Create Sample Data

Let's create our sensor data file with some intentionally bad data. Run the cell below to create the file.

**Figure 3.1: Creating the sample sensor data CSV file**

In [3]:
# ✏️ [EX1]
# Run this cell to create the sample data file.
# You do NOT need to modify this cell.

sample_data = """timestamp,sensor,value
2024-01-15 08:00,temp,22.5
2024-01-15 08:05,humidity,45.2
2024-01-15 08:10,temp,23.1
2024-01-15 08:15,pressure,1013.25
2024-01-15 08:20,temp,-999
2024-01-15 08:25,humidity,
2024-01-15 08:30,temp,22.8
2024-01-15 08:35,humidity,46.1
2024-01-15 08:40,pressure,abc
2024-01-15 08:45,temp,23.5
2024-01-15 08:50,humidity,44.8
2024-01-15 08:55,pressure,1012.80
2024-01-15 09:00,temp,24.0
2024-01-15 09:05,humidity,150.0
2024-01-15 09:10,pressure,1013.50
2024-01-15 09:15,temp,22.2
2024-01-15 09:20,humidity,43.5
2024-01-15 09:25,pressure,1011.90
2024-01-15 09:30,temp,
2024-01-15 09:35,humidity,47.3"""

with open("sensor_data.csv", "w") as f:
    f.write(sample_data)

print("\u2705 sensor_data.csv created successfully!")
print(f"File size: {len(sample_data)} bytes")

# Show the file contents
print("\n--- File Contents ---")
with open("sensor_data.csv", "r") as f:
    for i, line in enumerate(f):
        marker = ""
        # Mark potentially bad rows
        stripped = line.strip()
        if stripped.endswith(",") or ",," in stripped:
            marker = "  <-- EMPTY VALUE"
        elif "-999" in stripped:
            marker = "  <-- SUSPICIOUS VALUE"
        elif "abc" in stripped:
            marker = "  <-- NON-NUMERIC"
        elif "150.0" in stripped:
            marker = "  <-- OUT OF RANGE"
        print(f"  {i}: {stripped}{marker}")

✅ sensor_data.csv created successfully!
File size: 614 bytes

--- File Contents ---
  0: timestamp,sensor,value
  1: 2024-01-15 08:00,temp,22.5
  2: 2024-01-15 08:05,humidity,45.2
  3: 2024-01-15 08:10,temp,23.1
  4: 2024-01-15 08:15,pressure,1013.25
  5: 2024-01-15 08:20,temp,-999  <-- SUSPICIOUS VALUE
  6: 2024-01-15 08:25,humidity,  <-- EMPTY VALUE
  7: 2024-01-15 08:30,temp,22.8
  8: 2024-01-15 08:35,humidity,46.1
  9: 2024-01-15 08:40,pressure,abc  <-- NON-NUMERIC
  10: 2024-01-15 08:45,temp,23.5
  11: 2024-01-15 08:50,humidity,44.8
  12: 2024-01-15 08:55,pressure,1012.80
  13: 2024-01-15 09:00,temp,24.0
  14: 2024-01-15 09:05,humidity,150.0  <-- OUT OF RANGE
  15: 2024-01-15 09:10,pressure,1013.50
  16: 2024-01-15 09:15,temp,22.2
  17: 2024-01-15 09:20,humidity,43.5
  18: 2024-01-15 09:25,pressure,1011.90
  19: 2024-01-15 09:30,temp,  <-- EMPTY VALUE
  20: 2024-01-15 09:35,humidity,47.3


---
## Part 4: Step 2 — Read the Data

Now let's read the CSV file and parse it into a list of lists. Each inner list should contain `[timestamp, sensor, value_string]`.

**Figure 4.1: Example of reading and parsing CSV data**

In [4]:
# Example: How to read a CSV into a list of lists
# (This is a demonstration - you'll write your own version below)

with open("sensor_data.csv", "r") as f:
    header = f.readline().strip().split(",")
    print("Header:", header)
    
    # Read just the first 3 data rows as example
    for i in range(3):
        line = f.readline().strip()
        row = line.split(",")
        print(f"Row {i+1}: {row}")

Header: ['timestamp', 'sensor', 'value']
Row 1: ['2024-01-15 08:00', 'temp', '22.5']
Row 2: ['2024-01-15 08:05', 'humidity', '45.2']
Row 3: ['2024-01-15 08:10', 'temp', '23.1']


### Exercise 2: Read and Parse the CSV

Write code that reads `"sensor_data.csv"` and stores all data rows in a list called `raw_data`. Each element should be a list of 3 strings: `[timestamp, sensor, value]`.

**Expected output:**
```
Header: ['timestamp', 'sensor', 'value']
Total rows read: 20
First row: ['2024-01-15 08:00', 'temp', '22.5']
Last row: ['2024-01-15 09:35', 'humidity', '47.3']
```

<details><summary>💡 Hint</summary>

Read the first line as the header. Then loop through the remaining lines, `strip()` each line, `split(",")` it, and `append()` to `raw_data`.
</details>

In [5]:
# ✏️ [EX2]


---
### ⏱️ Checkpoint 2 of 5 — Validation (target 01:55)

Should malformed rows be validated before conversion? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [6]:
checkpoint_2_answer = ""  # enter your answer
check_answer(
    2, checkpoint_2_answer, 'yes',
    'Early validation prevents downstream failures.',
)


Checkpoint 2: enter your prediction, then run again.


False

---
## Part 5: Step 3 — Clean the Data

Now we need to validate each row and separate good data from bad data. A row is **bad** if:
- The value field is **empty**
- The value is **not a valid number**
- The value is **outside the valid range** for that sensor

### Validation Ranges Reminder

| Sensor | Min | Max |
|:---|:---|:---|
| `temp` | -50.0 | 60.0 |
| `humidity` | 0.0 | 100.0 |
| `pressure` | 800.0 | 1200.0 |

### Exercise 3: Validate a Single Row

Write a function called `validate_row(row)` that takes a list `[timestamp, sensor, value_string]` and returns a tuple `(is_valid, reason)`.

- If the row is valid: return `(True, "OK")`
- If the value is empty: return `(False, "empty value")`
- If the value is not a number: return `(False, "non-numeric value: 'xxx'")`
- If the value is out of range: return `(False, "out of range: xxx")`

**Expected output (test cases):**
```
['2024-01-15 08:00', 'temp', '22.5']     -> (True, 'OK')
['2024-01-15 08:25', 'humidity', '']      -> (False, 'empty value')
['2024-01-15 08:40', 'pressure', 'abc']   -> (False, "non-numeric value: 'abc'")
['2024-01-15 08:20', 'temp', '-999']      -> (False, 'out of range: -999.0')
['2024-01-15 09:05', 'humidity', '150.0'] -> (False, 'out of range: 150.0')
```

<details><summary>💡 Hint</summary>

1. Check if `row[2].strip()` is empty.
2. Try `float(row[2])` in a `try/except ValueError`.
3. Use a dictionary for ranges: `{"temp": (-50, 60), "humidity": (0, 100), "pressure": (800, 1200)}`.
4. Check if the value is within the range for the given sensor.
</details>

**Before indexing:** first check the row has exactly three string fields.
Then check required text, sensor membership, and timestamp format. Convert the value
with `float()`, catch conversion failures, test `math.isfinite(value)`, then test the
inclusive sensor range. `datetime.strptime(text, "%Y-%m-%d %H:%M")` validates a timestamp;
formatting it back with the same pattern lets you require the exact written format.
Test malformed rows, `light` as an unknown sensor, empty text, `nan`, `inf`, a valid
`temp=-10`, and boundary values before cleaning the full file.


In [7]:
# ✏️ [EX3]


### Exercise 4: Clean the Data

Using your `validate_row()` function, separate the data into two lists:
- `clean_data` — rows that passed validation
- `bad_data` — rows that failed, along with the reason

Print a summary of the cleaning results.

**Expected output:**
```
Data Cleaning Results
=====================
Total rows: 20
Clean rows: 15
Bad rows: 5

Bad rows detail:
  Row 5: 2024-01-15 08:20 | temp | -999 -> out of range: -999.0
  Row 6: 2024-01-15 08:25 | humidity |  -> empty value
  Row 9: 2024-01-15 08:40 | pressure | abc -> non-numeric value: 'abc'
  Row 14: 2024-01-15 09:05 | humidity | 150.0 -> out of range: 150.0
  Row 19: 2024-01-15 09:30 | temp |  -> empty value
```

<details><summary>💡 Hint</summary>

Loop through `raw_data` with `enumerate()` to track row numbers. Call `validate_row()` on each row. Append valid rows to `clean_data` and invalid ones (with reason) to `bad_data`.
</details>

In [8]:
# ✏️ [EX4]


---
## Part 6: Step 4 — Compute Statistics

Now let's compute statistics for each sensor type. We need to calculate:
- **Count** — how many valid readings
- **Min** — smallest value
- **Max** — largest value
- **Mean** — average value

### Exercise 5: Stats for One Sensor

Write a function called `compute_stats(data, sensor_name)` that takes the clean data list and a sensor name, and returns a dictionary with `count`, `min`, `max`, and `mean` for that sensor.

**Expected output (test):**
```
Stats for temp:
  count: 6
  min: 22.2
  max: 24.0
  mean: 23.02
```

<details><summary>💡 Hint</summary>

1. Filter `clean_data` to only include rows where `row[1] == sensor_name`.
2. Extract the values with `float(row[2])`.
3. Use `len()`, `min()`, `max()`, and `sum()/len()` on the values list.
4. Return a dictionary: `{"count": ..., "min": ..., "max": ..., "mean": ...}`.
</details>

**Hand check:** valid temperature values are `22.5, 23.1, 22.8, 23.5, 24.0, 22.2`; sum `138.1`, count `6`, mean `138.1/6 = 23.0166…`, displayed `23.02`. Humidity has `5` readings with mean `45.38`; pressure has `4` with mean `1012.8625`. Preserve precision internally.


In [9]:
# ✏️ [EX5]


### Exercise 6: Stats for All Sensors

Use your `compute_stats()` function to compute statistics for **all three sensors** (`temp`, `humidity`, `pressure`). Store the results in a dictionary called `all_stats`.

**Expected output:**
```
Sensor Statistics
=================

temp:
  Readings: 6
  Min: 22.20, Max: 24.00, Mean: 23.02

humidity:
  Readings: 5
  Min: 43.50, Max: 47.30, Mean: 45.38

pressure:
  Readings: 4
  Min: 1011.90, Max: 1013.50, Mean: 1012.86
```

<details><summary>💡 Hint</summary>

Create a list of sensor names: `["temp", "humidity", "pressure"]`. Loop through it, call `compute_stats()` for each, and store in `all_stats[sensor_name] = stats`.
</details>

In [10]:
# ✏️ [EX6]


---
### ⏱️ Checkpoint 3 of 5 — Decomposition (target 02:55)

Should reading, cleaning, and reporting be separate functions? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [11]:
checkpoint_3_answer = ""  # enter your answer
check_answer(
    3, checkpoint_3_answer, 'yes',
    'Small stages are easier to understand and test.',
)


Checkpoint 3: enter your prediction, then run again.


False

---
## Part 7: Step 5 — Generate Report

Now let's build a nicely formatted report string that summarizes everything.

### Exercise 7: Generate Summary Report

Write a function called `generate_report(all_stats, total_rows, clean_count, bad_count)` that returns a formatted string containing the full summary report.

The report should look like this:
```
======================================
    SENSOR LOG SUMMARY REPORT
======================================

DATA OVERVIEW
-------------
Total readings:   20
Valid readings:   15
Invalid readings: 5
Data quality:     75.0%

SENSOR: temp
-------------
  Readings: 6
  Min:      22.20
  Max:      24.00
  Mean:     23.02

SENSOR: humidity
-------------
  Readings: 5
  Min:      43.50
  Max:      47.30
  Mean:     45.38

SENSOR: pressure
-------------
  Readings: 4
  Min:      1011.90
  Max:      1013.50
  Mean:     1012.86

======================================
```

<details><summary>💡 Hint</summary>

Build the report string piece by piece using `+=` or a list of lines that you `"\n".join()` at the end. Use f-strings for formatting numbers.
</details>

**Empty-group formatting:** render `None` as `N/A`. If total rows are zero, report data quality as `N/A` rather than dividing by zero. Check that sensor counts sum to the clean-row count.


In [12]:
# ✏️ [EX7]


---
## Part 8: Step 6 — Write Clean Data

Now let's write the cleaned data to a new CSV file.

### Exercise 8: Write Clean CSV

Write the `clean_data` to a file called `"sensor_data_clean.csv"`. Include the header row. Then read the file back and print the first 5 lines to verify.

**Expected output:**
```
Clean data written to sensor_data_clean.csv (15 rows + header)

First 5 lines of clean file:
  timestamp,sensor,value
  2024-01-15 08:00,temp,22.5
  2024-01-15 08:05,humidity,45.2
  2024-01-15 08:10,temp,23.1
  2024-01-15 08:15,pressure,1013.25
```

<details><summary>💡 Hint</summary>

Open the file in write mode. Write the header `"timestamp,sensor,value\n"` first. Then loop through `clean_data` and write each row joined by commas.
</details>

In [13]:
# ✏️ [EX8]


---
## Part 9: Step 7 — Write Summary Report

Now write the summary report to a text file.

---
### ⏱️ Checkpoint 4 of 5 — Robustness (target 03:55)

Should one bad row crash the whole sensor report? Answer yes or no.

Enter a short answer in the next cell and run it. Retry after reviewing the
preceding examples if needed.


In [14]:
checkpoint_4_answer = ""  # enter your answer
check_answer(
    4, checkpoint_4_answer, 'no',
    'A robust pipeline records or skips bad rows deliberately.',
)


Checkpoint 4: enter your prediction, then run again.


False

### Exercise 9: Write Report to File

Write the report string (from Exercise 7) to a file called `"sensor_report.txt"`. Print a confirmation and then read back the file to verify.

**Expected output:**
```
Report written to sensor_report.txt

--- sensor_report.txt ---
======================================
    SENSOR LOG SUMMARY REPORT
======================================
...
```

<details><summary>💡 Hint</summary>

Simply open `"sensor_report.txt"` in write mode and call `file.write(report)` with the report string you generated in Exercise 7.
</details>

In [15]:
# ✏️ [EX9]


---
## Part 10: Putting It All Together

Now it's time to combine everything into a single, complete program.

### Exercise 10: The Complete Program

Write a `main()` function that does everything:
1. Reads `sensor_data.csv`
2. Validates and cleans the data
3. Computes statistics for each sensor
4. Generates a summary report
5. Writes `sensor_data_clean.csv`
6. Writes `sensor_report.txt`
7. Prints a final status message

Combine all your functions from previous exercises. Call `main()` at the end.

**Expected output:**
```
Sensor Log Summary Tool
=======================

[1/6] Reading sensor_data.csv...
      Found 20 data rows.

[2/6] Cleaning data...
      Valid: 15 | Invalid: 5

[3/6] Computing statistics...
      Processed 3 sensors.

[4/6] Generating report...
      Report ready.

[5/6] Writing sensor_data_clean.csv...
      Done.

[6/6] Writing sensor_report.txt...
      Done.

All tasks completed successfully!
```

<details><summary>💡 Hint</summary>

Copy your functions (`validate_row`, `compute_stats`, `generate_report`) into this cell. Then write a `main()` function that calls them in sequence. Use `print()` statements to show progress.
</details>

**Required robustness checks:** run once on the supplied 20-row dataset, then test an empty file, header-only file, wrong header, malformed rows, unknown sensors, invalid timestamps, nonfinite numbers, and valid negative temperatures. Each bad record receives a reason; it does not stop later valid records. Verify the persisted files by reading them back.

**Progress display:** the example above illustrates detailed logging. A shorter status message is also acceptable; the filenames, record counts and saved report statistics must agree.


In [16]:
# ✏️ [EX10]


---
### Checkpoint 5 of 5 — Practice reflection (target 04:45)

After Exercises 1–10, edit `practiced_exercises` in the next cell.
List only the exercise numbers whose results you have tested and whose steps you can explain.
Leave the list empty until you have done that work. This is your explicit self-report;
the tool does not inspect or grade your solution and does not count execution history.

**Türkçe:** Bu liste öz değerlendirmedir. Hücreyi çalıştırmak tek başına yeterli değildir;
sonucu kontrol et ve çözüm adımlarını açıklayabildiğinden emin ol.


In [17]:
# Add an exercise number only after testing and explaining your own work.
# Example: [1, 2] records your reflection about Exercises 1 and 2.
# Türkçe: Bu liste öz değerlendirmedir; kodunuzun doğruluğunu otomatik ölçmez.
practiced_exercises = []
exercise_checkpoint(5, practiced_exercises, expected=10)
show_progress_summary()


Practice self-report: 0/10 core exercises reviewed.
This is your reflection, not a correctness score or a grade.
Still to review: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
For each exercise: test the result, explain the steps, then compare with the worked solution.

My study feedback (this runtime)
1. Concept check: 0/1
2. Concept check: 0/1
3. Concept check: 0/1
4. Concept check: 0/1
5. Practice self-report: 0/10
These checks send no grading submission. Save your notebook to keep your work.


---
## 🌟 Optional Extension

Exercises 11–12 are optional enrichment. Stop here if the five-hour class has ended.


---
## Bonus Exercises

These are optional challenges for students who finish early.

### Exercise 11: BONUS — Date/Time Filtering (Challenge)

Add a feature to your program that filters the **cleaned data** by a time range. Write a function called `filter_by_time(data, start_time, end_time)` that returns only rows where the timestamp is between `start_time` and `end_time` (inclusive).

Since timestamps are in `YYYY-MM-DD HH:MM` format, you can compare them as strings (alphabetical comparison works correctly for this format).

**Expected output:**
```
Filtering: 2024-01-15 08:30 to 2024-01-15 09:00
Found 6 readings in time range.

Filtered data:
  2024-01-15 08:30 | temp      | 22.8
  2024-01-15 08:35 | humidity  | 46.1
  ...
```

<details><summary>💡 Hint</summary>

String comparison works for ISO-format dates: `"2024-01-15 08:30" >= start_time and "2024-01-15 08:30" <= end_time`. Filter the list using this condition on `row[0]`.
</details>
The 08:40 pressure record contains `abc` and is rejected before filtering. The six retained timestamps are 08:30, 08:35, 08:45, 08:50, 08:55, and 09:00. Validate the endpoints and require start ≤ end.


In [18]:
# ✏️ [EX11]


### Exercise 12: BONUS — Sensor Comparison (Challenge)

Write a function called `compare_sensors(all_stats)` that prints a comparison table showing all sensors side by side.

**Expected output:**
```
Sensor Comparison Table
=======================

Metric       temp        humidity    pressure   
------       ----        --------    --------   
Count        6           5           4          
Min          22.20       43.50       1011.90    
Max          24.00       47.30       1013.50    
Mean         23.02       45.38       1012.86    
Range        1.80        3.80        1.60       
```

The **range** is `max - min`.

<details><summary>💡 Hint</summary>

Use f-string formatting with fixed widths (e.g., `f"{value:<12}"`) to align columns. Loop through the metrics (count, min, max, mean) and for each metric, print the value for each sensor.
</details>

**Interpretation:** compare counts and processing quality across sensors, but do not rank temperature, humidity, and pressure means as if they had the same unit. Display N/A for an empty sensor group.


In [19]:
# ✏️ [EX12]


---
## What's Next?

Congratulations on completing **Computer Programming I**! You have come a long way from your first `print("Hello, World!")` to building a complete data processing tool.

Here is a summary of what you learned this semester:

| Week | Topic |
|:---|:---|
| 1 | Variables, Types & print() |
| 2 | Operators, f-strings & Type Conversion |
| 3 | Conditionals (if/elif/else) |
| 4 | for Loops & range() |
| 5 | while Loops, break & continue |
| 6 | Problem-Solving Patterns |
| 7 | Lists Fundamentals |
| 8 | 2D Lists & Nested Loops |
| 9 | String Processing |
| 10 | Functions: Basics |
| 11 | Scope & Mini-Library |
| 12 | Error Handling |
| 13 | File I/O & CSV |
| 14 | Mini Project: Sensor Log Summary |

### In Computer Programming II, you will explore:

- **Dictionaries and Sets** — more powerful data structures
- **Object-Oriented Programming (OOP)** — classes, objects, inheritance
- **Error Handling** — try/except, custom exceptions
- **Modules and Libraries** — NumPy, Pandas, Matplotlib
- **Working with APIs** — fetching data from the internet
- **Databases** — storing data with SQLite
- **Larger Projects** — putting it all together

Keep coding and keep learning. The best way to improve is to **practice every day**, even if it's just for 15 minutes.

> 💡 **Note:** If you enjoyed this course, consider exploring Python projects on your own: build a simple game, automate a daily task, or analyze data that interests you!

## Worked solutions and study support

Try each problem first. Then compare your reasoning and test cases with the complete
[Week 14 worked solutions](../solutions/Week_14_Solutions.ipynb).
The solution notebook includes every core exercise, optional exercise, and this week's bridge when present.

If you are using Colab, [open the published solution notebook](https://colab.research.google.com/github/ArifSolmaz/courses/blob/main/fall/cp1/solutions/Week_14_Solutions.ipynb).
Open it in a separate runtime. Running a solution first should not supply hidden variables to your own work.

**Türkçe:** Önce kendi çözümünü dene. Sonra adımları ve testleri karşılaştır; çözümü kapatıp farklı bir örneği kendin çöz.

[Simple course guide](../STUDY_GUIDE.md) · [All worked solutions](../solutions/README.md)
